In [13]:
%pip install -q chromadb torch sentence-transformers

In [14]:
import numpy as np
import pandas as pd
import os
import re
from html import unescape
from bs4 import BeautifulSoup
import chromadb
import torch
from sentence_transformers import SentenceTransformer

In [3]:
# path = kagglehub.dataset_download("ashking6393/faqs-csv")

# csv_path = os.path.join(path, "codebasics_faqs.csv")
df = pd.read_csv("https://raw.githubusercontent.com/sunlab-osu/covid-faq/master/data/FAQ_Bank.csv")

df.head()

,index,url,question,original answer,answer,answer wLink,source,language,type
0,0,https://www.azahcccs.gov/AHCCCS/AboutUs/covid1...,Does AHCCCS have a centralized resource for m...,"<dd><b>Answer: </b>Yes, the AHCCCS <a href=""/P...","Yes, the AHCCCS Medical Coding Resources webp...","Answer: Yes, the AHCCCS Medical Coding Resourc...",AHCCCS,en,Question
1,1,https://www.azahcccs.gov/AHCCCS/AboutUs/covid1...,Are there billing codes available for COVID-1...,<dd><b>Answer: </b>Yes. The Centers for Medica...,Yes. The Centers for Medicare & Medicaid Serv...,Answer: Yes. The Centers for Medicare & Medica...,AHCCCS,en,Question
2,2,https://www.azahcccs.gov/AHCCCS/AboutUs/covid1...,Will AHCCCS issue guidance regarding prior au...,<dd><b>Answer: </b>Prior authorization is not ...,Prior authorization is not permitted for COVI...,Answer: Prior authorization is not permitted f...,AHCCCS,en,Question
3,3,https://www.azahcccs.gov/AHCCCS/AboutUs/covid1...,Is there a claims modifier for services relat...,"<dd><b>Answer: </b>Yes, AHCCCS has designated ...","Yes, AHCCCS has designated the CR modifier to...","Answer: Yes, AHCCCS has designated the CR modi...",AHCCCS,en,Question
4,4,https://www.azahcccs.gov/AHCCCS/AboutUs/covid1...,Does AHCCCS cover testing for COVID-19?,"<dd><b>Answer: </b>Yes, AHCCCS covers COVID-19...","Yes, AHCCCS covers COVID-19 testing. HCPCS U0...","Answer: Yes, AHCCCS covers COVID-19 testing. H...",AHCCCS,en,Question


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15919 entries, 0 to 15918
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   index            15919 non-null  int64 
 1   url              15919 non-null  object
 2   question         15918 non-null  object
 3   original answer  15918 non-null  object
 4   answer           15885 non-null  object
 5   answer wLink     15918 non-null  object
 6   source           15919 non-null  object
 7   language         15919 non-null  object
 8   type             15919 non-null  object
dtypes: int64(1), object(8)
memory usage: 1.1+ MB


In [5]:
df.isna().sum()

,0
index,0
url,0
question,1
original answer,1
answer,34
answer wLink,1
source,0
language,0
type,0


In [6]:
def clean_text(text):
    text = str(text)
    if re.search(r"<[a-zA-Z/][^>]*>", text):
        text = BeautifulSoup(text, "html.parser").get_text(" ")

    text = unescape(text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [7]:
df["question"] = df["question"].astype(str).str.strip()
df["answer"] = df["answer"].astype(str).str.strip()

df_clean = df[["question","answer"]].copy()
df_clean = df_clean.dropna()

for col in ["question", "answer"]:
  df_clean[col] = df_clean[col].apply(clean_text)


df_clean = df_clean[
    (df_clean["question"] != "") &
    (df_clean["answer"] != "")
].copy()

df_clean = df_clean.drop_duplicates()

df_clean = df_clean.reset_index(drop=True)

df_clean.head()

,question,answer
0,Does AHCCCS have a centralized resource for me...,"Yes, the AHCCCS Medical Coding Resources webpa..."
1,Are there billing codes available for COVID-19...,Yes. The Centers for Medicare & Medicaid Servi...
2,Will AHCCCS issue guidance regarding prior aut...,Prior authorization is not permitted for COVID...
3,Is there a claims modifier for services relate...,"Yes, AHCCCS has designated the CR modifier to ..."
4,Does AHCCCS cover testing for COVID-19?,"Yes, AHCCCS covers COVID-19 testing. HCPCS U00..."


In [8]:
def create_document(row):
    return f"Question: {row['question']}\nAnswer: {row['answer']}"


In [9]:
documents = df_clean.apply(create_document, axis=1).tolist()
print(documents[0])

Question: Does AHCCCS have a centralized resource for medical coding resources related to COVID-19?
Answer: Yes, the AHCCCS Medical Coding Resources webpage includes a COVID-19 Medical Coding Information section and COVID-19 Emergency Medical Coding guidance.


In [10]:
def chunk_text(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))

        if end >= len(words):
            break

        start = end - overlap

    return chunks


In [11]:
chunks = []

for doc_id, document in enumerate(documents):
    for chunk_id, chunk in enumerate(chunk_text(document)):
        chunks.append({
            "id": f"faq_{doc_id}_chunk_{chunk_id}",
            "text": chunk,
            "faq_id": doc_id
        })

print("\nExample chunk:")
print(chunks[0]["text"])
# print(chunks[1]["text"])



Example chunk:
Question: Does AHCCCS have a centralized resource for medical coding resources related to COVID-19? Answer: Yes, the AHCCCS Medical Coding Resources webpage includes a COVID-19 Medical Coding Information section and COVID-19 Emergency Medical Coding guidance.


In [15]:
# client = chromadb.PersistentClient(path="./chroma_faq_db")

# embedding_function = chromadb.utils.embedding_functions.DefaultEmbeddingFunction()

# collection = client.get_or_create_collection(
#     name="faq_collection",
#     embedding_function=embedding_function,
#     metadata={"hnsw:space": "cosine"}
# )

# batch_size = 500

# for start in range(0, len(chunks), batch_size):
#     batch = chunks[start:start + batch_size]

#     collection.upsert(
#         ids=[item["id"] for item in batch],
#         documents=[item["text"] for item in batch],
#         metadatas=[{"faq_id": item["faq_id"]} for item in batch]
#     )

# print("Stored chunks:", collection.count())

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
model.half()

client = chromadb.PersistentClient(path="./chroma_faq_db")

collection = client.get_or_create_collection(
    name="faq_collection", metadata={"hnsw:space": "cosine"}
)

texts = [item["text"] for item in chunks]
embeddings = model.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

batch_size = 1000
for start in range(0, len(chunks), batch_size):
    batch = chunks[start : start + batch_size]
    batch_embeddings = embeddings[start : start + batch_size]

    collection.upsert(
        ids=[item["id"] for item in batch],
        documents=[item["text"] for item in batch],
        embeddings=batch_embeddings.tolist(),
        metadatas=[{"faq_id": item["faq_id"]} for item in batch],
    )

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/90 [00:00<?, ?it/s]

In [16]:
query = "How can I get more information?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

print("Query:", query)
print("\nTop 3 matches:\n")

for i, text in enumerate(results["documents"][0], start=1):
    distance = results["distances"][0][i - 1]

    print(f"Match {i} | Cosine distance: {distance:.4f}")
    print(text)
    print("-" * 60)


Query: How can I get more information?

Top 3 matches:

Match 1 | Cosine distance: 0.5467
Question: Provide more information if it’s requested. Answer: If you get a “Need More Information” message, follow the prompts to provide the necessary info. This will help ensure that you get your payment as quickly as possible! You may need to fill out information such as: Your bank account number Your bank’s routing number The type of account you’re providing information for (e.g., checking or savings) Additional security questions to verify your identity
------------------------------------------------------------
Match 2 | Cosine distance: 0.5493
Question: Where can I get more information? Answer: Further information about self-care can be found on the WHO website:4 More information about COVID-19 and sexual and reproductive health More information on COVID-19
------------------------------------------------------------
Match 3 | Cosine distance: 0.5601
Question: Where can I learn more about 

In [ ]:
df = pd.read_csv("test/test_covid_faq.csv")
for query in df["query"]:
    results = collection.query(
        query_texts=[query],
        n_results=3
    )

    print("Query:", query)
    print("\nTop 3 matches:\n")

    for i, text in enumerate(results["documents"][0], start=1):
        distance = results["distances"][0][i - 1]

        print(f"Match {i} | Cosine distance: {distance:.4f}")
        print(text)
        print("-" * 60)